# 5.1 — MuscleMimic Fullbody: Load a Policy

This is a minimal educational example for running a policy with a trajectory reference.

## 5.1.1 — Flow

1. Set a checkpoint + motion path.
2. Load model, trajectory, and policy.
3. Run rollout offscreen and preview MP4 inline.
4. (Optional) try interactive MuJoCo viewer.

No CLI wrappers are used.

> **Citation and data:** these tutorials use [MuscleMimic](https://github.com/amathislab/musclemimic) (Li et al., 2026, [arXiv:2603.25544](https://arxiv.org/abs/2603.25544)); please cite it when you use the models, checkpoints or retargeted motions. The motions are derived from [AMASS](https://amass.is.tue.mpg.de) (Mahmood et al., ICCV 2019, [arXiv:1904.03278](https://arxiv.org/abs/1904.03278)); the KIT clips (e.g. `KIT/314/walking_medium09_poses`) come from the KIT whole-body human motion database (Mandery et al., [2015](https://h2t.iar.kit.edu/pdf/Mandery2015a.pdf), [2016](https://doi.org/10.1109/TRO.2016.2572685)) and were retargeted with GMR (Araújo et al., 2025, [arXiv:2510.02252](https://arxiv.org/abs/2510.02252)). The datasets are for non-commercial research under the AMASS license. All references: the *References* page of the MyoSuite documentation.

In [ ]:
import os; _nb = globals().get("__vsc_ipynb_file__"); _nb and os.chdir(os.path.dirname(_nb))  # VS Code: work relative to this notebook

import os
# mm-10m-2 is restored on the CPU; a CUDA JAX plugin would otherwise pick the GPU.
os.environ.setdefault("JAX_PLATFORMS", "cpu")

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

CHECKPOINT_REF = "hf://amathislab/mm-10m-2"
MOTION_PATH = "KIT/314/walking_medium09_poses"
N_STEPS = 300
EVAL_SEED = 0
FRAME_SKIP = 5
VIDEO_PATH = Path("files/5.1/musclemimic_fullbody_policy_preview.mp4")
REQUESTED_WIDTH = 960
REQUESTED_HEIGHT = 540
RUN_INTERACTIVE_VIEWER = False  # optional live viewer; needs a working display (not SSH without X)
policy = None
model = None
clip = None
checkpoint_root = None
_MM_OK = False

try:
    import mujoco
    from IPython.display import Markdown, Video, display

    from myosuite.integrations.musclemimic.fullbody_checkpoint_io import resolve_checkpoint_ref
    from myosuite.integrations.musclemimic.fullbody_local_policy import (
        FullbodyObsAdapter,
        LocalPolicyRunner,
        has_local_policy_artifacts,
        load_local_policy_artifacts,
        read_checkpoint_config_metadata,
    )
    from myosuite.integrations.musclemimic.fullbody_model import (
        compile_musclemimic_fullbody_mjmodel,
        default_musclemimic_fullbody_config,
    )
    from myosuite.integrations.musclemimic.model_versions import checkpoint_models_version
    from myosuite.core.trajectory_io import load_motion_clip, resolve_motion_path
    from myosuite.integrations.musclemimic.hf_demo_cache import setup_demo_for_myo_fullbody
except Exception as e:
    print(f"[SKIP] MuscleMimic extras not available: {e}")
    print("Install with: pip install -e '.[musclemimic]'")
else:
    _MM_OK = True
    print("Checkpoint:", CHECKPOINT_REF)
    print("Motion:", MOTION_PATH)
    print("Steps:", N_STEPS)

In [ ]:
# 1) Resolve checkpoint (gated / large download — skip cleanly if unavailable)
try:
    checkpoint = resolve_checkpoint_ref(CHECKPOINT_REF)
    checkpoint_root = checkpoint.local_path
    if not has_local_policy_artifacts(checkpoint_root):
        raise RuntimeError("Checkpoint is missing local policy artifacts (train_state/config).")
except Exception as e:
    print(f"[SKIP] Could not load MuscleMimic checkpoint {CHECKPOINT_REF!r}: {e}")
    print("Request Hugging Face access or set CHECKPOINT_REF to a local path.")
    checkpoint_root = None
    model = None
    clip = None
    policy = None
else:
    # 2) Build full-body MuJoCo model
    cfg = default_musclemimic_fullbody_config()
    # The physics the checkpoint was trained on (musclemimic_models 1.0.5 for mm-10m-2).
    cfg.model_version = checkpoint_models_version(CHECKPOINT_REF)
    model, _spec, xml_path = compile_musclemimic_fullbody_mjmodel(cfg)

    # 3) Load trajectory clip
    setup_demo_for_myo_fullbody()  # downloads the demo clips once; later runs reuse the cache
    motion_file = resolve_motion_path(MOTION_PATH, env_name="MyoFullBody")
    clip = load_motion_clip(motion_file, expected_nq=model.nq, expected_nv=model.nv)

    # 4) Build obs adapter from checkpoint goal_params + load policy
    cfg_meta = read_checkpoint_config_metadata(checkpoint_root)
    goal_params = (
        cfg_meta.get("experiment", {})
        .get("env_params", {})
        .get("goal_params", {})
    )
    obs_adapter = FullbodyObsAdapter(model=model, clip=clip, goal_params=goal_params)

    artifacts = load_local_policy_artifacts(checkpoint_root)
    policy = LocalPolicyRunner(
        artifacts=artifacts,
        stochastic=False,
        seed=EVAL_SEED,
        frame_skip=FRAME_SKIP,
        obs_adapter=obs_adapter,
    )

    print("Resolved checkpoint:", checkpoint_root)
    print("Model dims (nq, nv, nu):", (model.nq, model.nv, model.nu))
    print("Motion file:", motion_file)
    print("Frames in clip:", int(clip.qpos.shape[0]))

In [ ]:
if policy is None:
    print('[SKIP] no MuscleMimic policy loaded')
else:
    from myosuite.utils.video_io import write_video
    from myosuite.viz.rollout_video import record_rollout, tracking_camera

    data = clip.to_mjdata(model)  # start at the clip's first frame
    steps = min(N_STEPS, clip.num_frames)
    qpos_l2_errors: list[float] = []

    def step(i: int) -> None:
        """One control step of the policy, then the tracking error to the clip."""
        policy.step(model, data, policy.action_for(data, clip, i))
        qpos_l2_errors.append(float(np.linalg.norm(data.qpos - clip.qpos[i])))

    frames = record_rollout(
        model, data, step, steps,
        width=REQUESTED_WIDTH, height=REQUESTED_HEIGHT, camera=tracking_camera(model),
    )
    VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)
    fps = round(1.0 / (model.opt.timestep * FRAME_SKIP))  # one frame per control step
    write_video(VIDEO_PATH, frames, fps)

    display(Markdown(
        f"""
### Offscreen rollout complete
- Steps: `{len(frames)}`
- Mean qpos L2 error: `{np.mean(qpos_l2_errors):.6f}`
- Max qpos L2 error: `{np.max(qpos_l2_errors):.6f}`
- Video: `{VIDEO_PATH}`
"""
    ))
    display(Video(str(VIDEO_PATH), embed=True, html_attributes="controls loop"))

In [ ]:
if policy is None:
    print('[SKIP] no MuscleMimic policy loaded')
elif not RUN_INTERACTIVE_VIEWER:
    print('[SKIP] interactive viewer (set RUN_INTERACTIVE_VIEWER = True to try it)')
else:
    # Optional live preview. It needs a working display: without one, MuJoCo aborts
    # the whole Python process (a kernel crash) instead of raising an exception.
    import mujoco.viewer

    data = clip.to_mjdata(model)
    preview_steps = min(int(N_STEPS), 200)
    with mujoco.viewer.launch_passive(model, data) as viewer:
        for i in range(preview_steps):
            action = policy.action_for(data, clip, i)
            policy.step(model, data, action)
            viewer.sync()
    print(f"Interactive preview completed for {preview_steps} steps.")

## 5.1.2 — Troubleshooting

- If checkpoint loading fails, install `orbax-checkpoint` in this environment.
- If MP4 export fails, install `imageio`.
- If interactive viewer fails on macOS, prefer offscreen rendering in notebooks.

To try another trajectory, only change `MOTION_PATH` and re-run from the load cell.